### 1) Load the `cat-facts` dataset

The dataset contains a list of cat facts that will be used as chunks in the indexing phrase.

This first step reads every line from the `cat-facts` file into memory.
Each line is treated as one knowledge chunk that will later be embedded and searched.




In [5]:
dataset = []
with open('cat-facts', 'r' , encoding='UTF-8') as file:
  dataset = file.readlines()
  print(f'Loaded {len(dataset)} entries')

Loaded 150 entries


### 2) Import Ollama and configure the models

Here we import `ollama` and define the embedding model and the language model.
The embedding model converts text into vectors, while the language model generates the final answer.

We will use the embedding model from ollama to convert each chunk into an embedding vector, then store the chunk and its corresponding vector in a list.


In [7]:
import ollama

EMBEDDING_MODEL = 'hf.co/CompendiumLabs/bge-base-en-v1.5-gguf'
LANGUAGE_MODEL = 'hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF'

# Each element in the VECTOR_DB will be a tuple (chunk, embedding)
# The embedding is a list of floats, for example: [0.1, 0.04, -0.34, 0.21, ...]
VECTOR_DB = []

def add_chunk_to_database(chunk):
  embedding = ollama.embed(model=EMBEDDING_MODEL, input=chunk)['embeddings'][0]
  VECTOR_DB.append((chunk, embedding))

### 3) Build the vector database from each cat fact

This loop turns every cat fact into an embedding and stores it in `VECTOR_DB`.
That database is the retrieval memory of the RAG system.


In [8]:
for i, chunk in enumerate(dataset):
  add_chunk_to_database(chunk)
  print(f'Added chunk {i+1}/{len(dataset)} to the database')


Added chunk 1/150 to the database
Added chunk 2/150 to the database
Added chunk 3/150 to the database
Added chunk 4/150 to the database
Added chunk 5/150 to the database
Added chunk 6/150 to the database
Added chunk 7/150 to the database
Added chunk 8/150 to the database
Added chunk 9/150 to the database
Added chunk 10/150 to the database
Added chunk 11/150 to the database
Added chunk 12/150 to the database
Added chunk 13/150 to the database
Added chunk 14/150 to the database
Added chunk 15/150 to the database
Added chunk 16/150 to the database
Added chunk 17/150 to the database
Added chunk 18/150 to the database
Added chunk 19/150 to the database
Added chunk 20/150 to the database
Added chunk 21/150 to the database
Added chunk 22/150 to the database
Added chunk 23/150 to the database
Added chunk 24/150 to the database
Added chunk 25/150 to the database
Added chunk 26/150 to the database
Added chunk 27/150 to the database
Added chunk 28/150 to the database
Added chunk 29/150 to the dat

### 4) Define the cosine similarity function

Cosine similarity measures how close two vectors are.
The closer the vectors are, the more relevant a chunk is considered for the user query.

In [9]:
def cosine_similarity(a, b):
  dot_product = sum([x * y for x, y in zip(a, b)])
  norm_a = sum([x ** 2 for x in a]) ** 0.5
  norm_b = sum([x ** 2 for x in b]) ** 0.5
  return dot_product / (norm_a * norm_b)

### 5) Retrieve the most relevant chunks for a query

This function embeds the query, compares it with all stored chunks, and returns the most similar ones.
Those retrieved chunks will be sent to the language model as context.

In [10]:
def retrieve(query, top_n=3):
  query_embedding = ollama.embed(model=EMBEDDING_MODEL, input=query)['embeddings'][0]
  # temporary list to store (chunk, similarity) pairs
  similarities = []
  for chunk, embedding in VECTOR_DB:
    similarity = cosine_similarity(query_embedding, embedding)
    similarities.append((chunk, similarity))
  # sort by similarity in descending order, because higher similarity means more relevant chunks
  similarities.sort(key=lambda x: x[1], reverse=True)
  # finally, return the top N most relevant chunks
  return similarities[:top_n]

### 6) Ask a question and inspect the retrieved knowledge

Now we test the retrieval step with a user question.
The notebook prints the most relevant facts so you can verify whether the search is working correctly.


In [42]:
input_query = input('Ask me a question: ')
retrieved_knowledge = retrieve(input_query)

print('Retrieved knowledge:')
for chunk, similarity in retrieved_knowledge:
  print(f' - (similarity: {similarity:.2f}) {chunk}')

instruction_prompt = f'''You are a helpful chatbot.
Use only the following pieces of context to answer the question. Say I don't know when you not find the response. Don't make up any new information:
{'\n'.join([f' - {chunk}' for chunk, similarity in retrieved_knowledge])}
'''

Retrieved knowledge:
 - (similarity: 0.67) In Holland’s embassy in Moscow, Russia, the staff noticed that the two Siamese cats kept meowing and clawing at the walls of the building. Their owners finally investigated, thinking they would find mice. Instead, they discovered microphones hidden by Russian spies. The cats heard the microphones when they turned on.

 - (similarity: 0.66) The group of words associated with cat ( catt, cath, chat, katze ) stem from the Latin catus , meaning domestic cat, as opposed to feles , or wild cat.

 - (similarity: 0.64) Contrary to popular belief, the cat is a social animal. A pet cat will respond and answer to speech, and seems to enjoy human companionship.



### 7) Generate and stream the chatbot response

In the final step, the retrieved context is passed to the language model.
The response is streamed back so you can see the answer appear in real time.


In [43]:
stream = ollama.chat(
  model=LANGUAGE_MODEL,
  messages=[
    {'role': 'system', 'content': instruction_prompt},
    {'role': 'user', 'content': input_query},
  ],
  stream=True,
)

# print the response from the chatbot in real-time
print('Chatbot response:')
for chunk in stream:
  print(chunk['message']['content'], end='', flush=True)


Chatbot response:
No, it's not that simple. Cats don't use their language in the same way as humans do. While cats can make various sounds to communicate with each other, meowing and purring are more a form of emotional expression than a linguistic system.

Cats primarily use vocalizations like:

* Meowing: often used for attention-seeking, communication, or expressing emotions
* Purring: a self-soothing behavior that helps them relax and feel calm
* Hissing: a warning signal to alert others to potential danger
* Growling: a low-pitched sound that can be threatening or aggressive

Their language is more like a complex system of body language, vocalizations, and scent markings than words. So, while it's fun to imagine cats speaking human languages, it's not quite how they communicate with each other!